[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter17_convolutional_neural_networks/notebook.ipynb)

# Chapter 17: Convolutional Neural Networks

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

Every network in this book so far has been made of dense layers, in which every input connects to every unit. For measurements of a flower that is exactly right, because petal length and sepal width are different things and each deserves its own weight. Pictures are different, and this chapter is about why.

In 1988 a young French researcher named Yann LeCun, who had worked with Geoffrey Hinton in Toronto, joined AT&T Bell Laboratories in Holmdel, New Jersey. The US Postal Service had a practical problem: millions of envelopes with handwritten zip codes, and people reading them. In 1989 LeCun's group published a network trained by backpropagation that read those digits with an error rate of about one percent. Its layers were not dense. Each unit looked only at a small patch of the image, and the same small set of weights was slid across the whole picture.

That idea, refined over the next decade into a network called LeNet and published in full in 1998, went on to read millions of bank cheques a day. Fourteen years later it was the heart of AlexNet, the network that opened Chapter 1.

This chapter is a first look, and Book 7 is where images get the full treatment. Before you read on, try to guess the answers to these four questions.

1. **If a digit moves one pixel to the right, how different does it look to a dense layer?**
2. **Why might one small detector, reused everywhere, beat a separate weight for every pixel?**
3. **What could a network gain by deliberately throwing away some detail?**
4. **Could a network trained on one task give a head start on another?**

## Why Dense Layers Struggle with Images

The data for this chapter ships inside scikit-learn, so nothing needs downloading. It is a set of 1,797 handwritten digits, scanned and then shrunk by counting the inked pixels in each 4 by 4 block, which is why every image is only 8 by 8 pixels with values from 0 to 16.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
print(digits.images.shape, digits.images.min(), digits.images.max())

Split them the usual way and scale the pixels to run from 0 to 1. The extra dimension added by `unsqueeze(1)` is the number of **channels**, which is one for a grey image and three for a colour one. PyTorch expects images shaped as batch, channels, height, width.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    digits.images / 16.0, digits.target, test_size=0.25,
    random_state=7, stratify=digits.target)
Xtr = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
Xte = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
ytr, yte = torch.tensor(y_train), torch.tensor(y_test)
print(Xtr.shape, Xte.shape)

Now the problem. Take one test digit and slide it one pixel to the right.

In [ ]:
def shift(images, dx):
    out = torch.zeros_like(images)
    out[..., dx:] = images[..., :-dx]
    return out

digit, moved = Xte[2, 0], shift(Xte[2:3], 1)[0, 0]
print("label", yte[2].item())
print((digit != moved).sum().item(), "of 64 pixels changed")

> 🖼️ **Figure 17.1:** The same handwritten 6 before and after a one pixel move, drawn from the real pixel values. To you they are the same digit. To a dense layer, which gives every pixel position its own weight, 44 of the 64 inputs are now different.

A dense layer has no idea that pixel 12 and pixel 13 are neighbours. It has a weight for each position, so a stroke that falls one column further right lands on entirely different weights. To recognise a 6 anywhere, it must effectively learn the 6 separately at every position it might appear.

The second problem is size. These digits are tiny. A typical photograph fed to a modern image network is 224 by 224 pixels in three colours.

In [ ]:
pixels = 224 * 224 * 3
print(pixels, "inputs")
print(pixels * 1000 + 1000, "weights in one dense layer of 1000 units")
conv = nn.Conv2d(3, 64, kernel_size=3)
print(sum(p.numel() for p in conv.parameters()), "weights in 64 small filters")

A hundred and fifty million weights for a single dense layer, against under two thousand for a convolutional layer that, as the next section shows, does something more useful. The convolutional layer's count does not even depend on the size of the image.

## The Convolution, by Hand

A **convolution** slides a small grid of weights, called a **filter** or **kernel**, across the image. At each position it multiplies the patch underneath by the filter, number by number, and adds the results up. That sum becomes one number in the output.

Build an image you can check by eye: six by six, dark on the left half and bright on the right, so there is one vertical edge down the middle. The filter below is a vertical edge detector. Its left column is $-1$, its right column is $+1$, so it responds when the right side of a patch is brighter than the left.

In [ ]:
image = torch.zeros(6, 6)
image[:, 3:] = 1.0
kernel = torch.tensor([[-1., 0., 1.],
                       [-1., 0., 1.],
                       [-1., 0., 1.]])

by_hand = torch.zeros(4, 4)
for row in range(4):
    for col in range(4):
        patch = image[row:row + 3, col:col + 3]
        by_hand[row, col] = (patch * kernel).sum()
for line in by_hand.tolist():
    print([int(v) for v in line])

A three by three window fits into a six by six image in four positions across and four down, so the output is four by four. Where the window sits entirely in the dark or entirely in the bright half, the left and right columns cancel and the answer is 0. Where it straddles the edge, the three bright pixels under the $+1$ column survive and the answer is 3. The output is a map of where the vertical edge is, which is why it is called a **feature map**.

> 🖼️ **Figure 17.2:** One step of a convolution. The red window on the image covers columns 2 to 4 of rows 0 to 2; multiplying it by the filter and adding gives the single red-outlined 3 in the feature map. Sliding the window everywhere fills in the rest.

Now the trust check this book has used since Chapter 9. PyTorch's `F.conv2d` should give exactly the same grid.

In [ ]:
by_torch = F.conv2d(image.view(1, 1, 6, 6), kernel.view(1, 1, 3, 3))
for line in by_torch.view(4, 4).tolist():
    print([int(v) for v in line])
print(torch.equal(by_hand, by_torch.view(4, 4)))

Identical. One small point for later reading: mathematicians define convolution with the filter flipped, and what PyTorch and every other deep learning library actually compute is the unflipped version. Since the filter's weights are learned anyway, the difference never matters in practice.

That last sentence is the whole idea of a **convolutional layer**. Here you wrote the edge detector by hand. In a network the nine numbers start random and are learned by backpropagation, and a layer learns many filters at once, each producing its own feature map. Because the same nine weights are used at every position, a stroke is detected wherever it appears. That is the answer to the shifting problem, and to the size problem too.

## Pooling and Feature Maps

A feature map says where each feature was found, often more precisely than needed. **Max pooling** keeps the strongest response in each small block and throws the rest away. With two by two blocks it halves the height and the width.

In [ ]:
fmap = torch.tensor([[1., 5., 0., 2.],
                     [3., 2., 1., 4.],
                     [0., 1., 7., 0.],
                     [2., 0., 3., 1.]])
pooled = F.max_pool2d(fmap.view(1, 1, 4, 4), 2).view(2, 2)
for line in pooled.tolist():
    print([int(v) for v in line])

Each output is the largest of four: 5 from the top left block, 4 from the top right, 2 and 7 from the bottom pair. You can check all four by eye. Pooling keeps the fact that a feature was found somewhere in a region and drops exactly where, which buys a little tolerance to small shifts and cuts the amount of work for every layer after it.

Stack the pieces and a pattern emerges. Convolution finds simple features, pooling shrinks the map, the next convolution combines simple features into larger ones, and so on, which is Chapter 1's ladder of edges to shapes to objects made concrete. Trace the shapes through a small network for the digits.

In [ ]:
torch.manual_seed(0)
cnn = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64, 10))

h = Xtr[:5]
for layer in cnn:
    h = layer(h)
    if not isinstance(layer, nn.ReLU):
        print(type(layer).__name__, list(h.shape))

The `padding=1` adds a border of zeros so that a three by three filter can sit over the edge pixels, which keeps the map at 8 by 8 instead of shrinking it to 6 by 6. Five images go in as one channel of 8 by 8; the first convolution turns each into eight feature maps; pooling halves them to 4 by 4; the second convolution makes sixteen maps; pooling halves again to 2 by 2; and flattening gives $16 \times 2 \times 2 = 64$ numbers per image for an ordinary dense layer to turn into ten class scores. Figure 17.3 draws the same journey.

> 🖼️ **Figure 17.3:** The digit network stage by stage. The maps get smaller and more numerous as the features they hold get larger and more abstract.

## A Small Network That Sees

Now compare it with a dense network of a similar size on the real task. The training function is the familiar one from Chapter 12, with one addition: after training it measures accuracy on the test digits as written, then shifted one and two pixels to the right.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def fit(net, epochs=30, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    for epoch in range(epochs):
        net.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(net(xb), yb).backward()
            opt.step()
    net.eval()
    with torch.no_grad():
        accuracy = lambda X: (net(X).argmax(1) == yte).float().mean().item()
        return [round(accuracy(X), 3) for X in [Xte, shift(Xte, 1), shift(Xte, 2)]]

In [ ]:
torch.manual_seed(0)
dense = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(),
                      nn.Linear(64, 10))
for name, net in [("dense", dense), ("cnn", cnn)]:
    count = sum(p.numel() for p in net.parameters())
    print(name, count, fit(net))

Three things to read. On the digits as written, both networks are excellent and practically tied. The convolutional network does it with 1,898 weights against the dense network's 4,810, so about forty percent as many. And when the digits move, the dense network falls apart: one pixel costs it nearly half its accuracy, two pixels leave it at 0.2. The convolutional network loses ground too, but holds a clear lead at both shifts.

It does not shrug the shift off entirely, and it is worth knowing why. The filters are shared across positions, but the final dense layer still gives each of the 64 flattened numbers its own weight, and on images this small a shift of two pixels pushes a quarter of the digit off the edge. Real image systems close the remaining gap with deeper stacks of convolution and pooling and by training on shifted, rotated and rescaled copies of their images, a technique called **data augmentation** that Book 7 covers properly. This result is not a lucky seed either; one of the exercises asks you to confirm that.

## Standing on Other Networks' Shoulders: a Note on Pretrained Models

The first layers of any network trained on photographs end up learning much the same thing: edge detectors, colour blobs, simple textures. That suggests a shortcut. Take a network someone else trained for days on a million images, keep its lower layers as they are, and train only a new final layer for your own task. This is **transfer learning**, and the trained network you start from is a **pretrained model**.

The mechanics are only two steps, and you can try them without downloading anything by treating a digit network as the pretrained one. Train it on digits 0 to 4, freeze its convolutional layers, and attach a fresh final layer for digits 5 to 9, using only fifty labelled examples of the new digits.

In [ ]:
low = ytr < 5
torch.manual_seed(0)
base = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten())
print(int(low.sum()), "training digits from 0 to 4")
print(list(base(Xtr[:5]).shape))

The `base` is the convolutional network without its last layer, so it ends at the 64 flattened numbers. That cut point is the usual one: everything up to it is the reusable part, and the final dense layer is the task specific part.

In [ ]:
def train_on(net, X, y, epochs, params, seed=0):
    opt = torch.optim.Adam(params, lr=0.01)
    loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    for epoch in range(epochs):
        net.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(net(xb), yb).backward()
            opt.step()
    net.eval()

old = nn.Sequential(base, nn.Linear(64, 5))
train_on(old, Xtr[low], ytr[low], 30, old.parameters())

`train_on` is the familiar loop with one change: it takes the list of weights to update as an argument, rather than always using every weight in the network. That one argument is what makes the next step possible. With the network trained on the five low digits, freeze its body and give it a new head.

In [ ]:
for p in base.parameters():
    p.requires_grad = False
torch.manual_seed(0)
new_head = nn.Linear(64, 5)
reused = nn.Sequential(base, new_head)
print(sum(p.numel() for p in reused.parameters() if p.requires_grad), "trainable")

Setting `requires_grad` to `False` is the freeze, the same switch Chapter 9 introduced, now used to tell autograd to leave those weights alone. Only the 325 weights of the new head will learn. Now give it fifty examples of digits 5 to 9 and compare with the same small network trained from scratch on those same fifty.

In [ ]:
high = torch.nonzero(ytr >= 5).flatten()
few = high[torch.randperm(len(high), generator=torch.Generator().manual_seed(7))[:50]]
X_few, y_few = Xtr[few], ytr[few] - 5
test_high = yte >= 5

train_on(reused, X_few, y_few, 100, new_head.parameters())
torch.manual_seed(0)
scratch = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64, 5))
train_on(scratch, X_few, y_few, 100, scratch.parameters())

with torch.no_grad():
    for name, net in [("reused", reused), ("scratch", scratch)]:
        hits = net(Xte[test_high]).argmax(1) == yte[test_high] - 5
        print(name, round(hits.float().mean().item(), 3))

The reused network comes out ahead, 0.946 to 0.924, while training only about a fifth as many weights. Do not trust a two point lead from a single run, though. Repeating the comparison with other random choices of the fifty examples, and with 25 or 100 examples instead of 50, the two trade places with no consistent winner: across fifteen such runs, training from scratch came out ahead eight times and the reused network five, with two ties.

The reason is the source. These frozen features were learned from 675 tiny pictures of five digits, which is not enough variety to be reliably more useful than what a few dozen examples can teach a small network from scratch.

Transfer learning earns its reputation when the source is vast and varied and the new task is hard. A network pretrained on over a million photographs of a thousand kinds of object has filters that are useful for almost any picture, and with them a few hundred labelled X-rays or product photographs can be enough to build something good. Libraries ship such networks ready to use; `torchvision.models.resnet18`, for example, comes with about 11.7 million weights already trained, at the cost of a download. Book 7 uses exactly that, and the two steps it takes are the ones you just wrote: freeze the body, replace the head.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Turn the edge detector.** Transpose `kernel` so it detects horizontal edges, run it over `image`, and explain why the output is all zeros.
2. **Check the seed.** Rerun the dense against convolutional comparison with seeds 1, 2 and 3 and confirm that the convolutional network keeps its lead on shifted digits.
3. **Augment the data.** Add shifted copies of the training digits to `Xtr` and retrain both networks. How much of the shift problem does that fix for each?
4. **Look at a learned filter.** Print `cnn[0].weight[0, 0]` after training and see whether it resembles an edge detector.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **Why does a one pixel shift hurt a dense layer so badly?**
2. **Why does a convolutional layer have so few weights, and why does its count not depend on the image size?**
3. **A 3 by 3 filter slides over an 8 by 8 image with no padding. What size is the output?**
4. **What does max pooling keep and what does it throw away?**
5. **What are the two steps of transfer learning?**
6. **Why did transfer learning not reliably help on the digits in this chapter?**

### Answers

1. Because it gives every pixel position its own weight, so a moved stroke lands on different weights. Here 44 of 64 inputs changed.
2. Because one small filter is reused at every position, so its weights are counted once. A bigger image means more positions, not more weights.
3. 6 by 6, since the window fits in six positions across and six down.
4. It keeps the strongest response in each block and throws away exactly where in the block it occurred.
5. Freeze the pretrained layers by setting `requires_grad` to `False`, then replace and train only the final layer for the new task.
6. It helped in the one run shown but not consistently across runs, because the frozen features came from too little and too narrow a source, 675 tiny images of five digits, to be reliably more useful than a small network trained from scratch on the new data.

## What Comes Next

You can now explain why images need a different kind of layer, compute a convolution and a pooling step by hand, build and read the shapes of a small convolutional network, and freeze and reuse part of a trained network.

- **Chapter 18, Sequence Models and Embeddings,** turns from data laid out in space to data laid out in time, and from pixels to symbols.
- **Book 7, Computer Vision and Multimodal AI,** takes this chapter's ideas to full size photographs, pretrained networks and data augmentation.

Chapter 18 starts with a question convolution cannot answer: how should a network read a sentence whose meaning depends on a word it saw ten words ago?